In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("DS_assignment_data - Copy (1) (1).csv")

print(df.head())
print(df.shape)

   UNIQUEID  DISBURSED_AMOUNT  ASSET_COST    LTV  BRANCH_ID  \
0    420825             50578       58400  89.55         67   
1    537409             47145       65550  73.23         67   
2    417566             53278       61360  89.63         67   
3    624493             57513       66113  88.48         67   
4    539055             52378       60300  88.39         67   

   CURRENT_PINCODE_ID DATE_OF_BIRTH EMPLOYMENT_TYPE DISBURSAL_DATE  \
0                1441    01/01/1984        Salaried     03/08/2018   
1                1502    31/07/1985   Self employed     26/09/2018   
2                1497    24/08/1985   Self employed     01/08/2018   
3                1501    30/12/1993   Self employed     26/10/2018   
4                1495    09/12/1977   Self employed     26/09/2018   

   MOBILENO_AVL_FLAG  ...  SEC_SANCTIONED_AMOUNT  SEC_DISBURSED_AMOUNT  \
0                  1  ...                      0                     0   
1                  1  ...                      0    

In [3]:
df["DATE_OF_BIRTH"] = pd.to_datetime(df["DATE_OF_BIRTH"], errors="coerce", dayfirst=True)
df["DISBURSAL_DATE"] = pd.to_datetime(df["DISBURSAL_DATE"], errors="coerce", dayfirst=True)

df["AGE_YEARS"] = (df["DISBURSAL_DATE"] - df["DATE_OF_BIRTH"]).dt.days / 365.25

print(df["AGE_YEARS"].describe())

count    5000.000000
mean       34.781669
std         9.870244
min        18.078029
25%        26.540726
50%        32.896646
75%        41.735797
max        64.010951
Name: AGE_YEARS, dtype: float64


In [4]:
clf_df = df[
    [
        "LOAN_DEFAULT",
        "PERFORM_CNS_SCORE",
        "ASSET_COST",
        "LTV",
        "EMPLOYMENT_TYPE",
        "DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS",
        "NO_OF_INQUIRIES",
        "AGE_YEARS"
    ]
].copy()

clf_df["EMPLOYMENT_TYPE"] = clf_df["EMPLOYMENT_TYPE"].fillna("Unknown")
clf_df = clf_df.dropna()

print(clf_df.isna().sum())
print(clf_df.shape)

LOAN_DEFAULT                           0
PERFORM_CNS_SCORE                      0
ASSET_COST                             0
LTV                                    0
EMPLOYMENT_TYPE                        0
DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS    0
NO_OF_INQUIRIES                        0
AGE_YEARS                              0
dtype: int64
(5000, 8)


In [5]:
clf_df = pd.get_dummies(clf_df, drop_first=True)

print(clf_df.head())

   LOAN_DEFAULT  PERFORM_CNS_SCORE  ASSET_COST    LTV  \
0             0                  0       58400  89.55   
1             1                598       65550  73.23   
2             0                  0       61360  89.63   
3             1                305       66113  88.48   
4             1                  0       60300  88.39   

   DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS  NO_OF_INQUIRIES  AGE_YEARS  \
0                                    0                0  34.587269   
1                                    1                0  33.155373   
2                                    0                0  32.936345   
3                                    0                1  24.821355   
4                                    0                1  40.796715   

   EMPLOYMENT_TYPE_Self employed  EMPLOYMENT_TYPE_Unknown  
0                          False                    False  
1                           True                    False  
2                           True                    Fals

In [6]:
X = clf_df.drop("LOAN_DEFAULT", axis=1)
y = clf_df["LOAN_DEFAULT"]

print(X.head())
print(y.value_counts())

   PERFORM_CNS_SCORE  ASSET_COST    LTV  DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS  \
0                  0       58400  89.55                                    0   
1                598       65550  73.23                                    1   
2                  0       61360  89.63                                    0   
3                305       66113  88.48                                    0   
4                  0       60300  88.39                                    0   

   NO_OF_INQUIRIES  AGE_YEARS  EMPLOYMENT_TYPE_Self employed  \
0                0  34.587269                          False   
1                0  33.155373                           True   
2                0  32.936345                           True   
3                1  24.821355                           True   
4                1  40.796715                           True   

   EMPLOYMENT_TYPE_Unknown  
0                    False  
1                    False  
2                    False  
3                 

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (4000, 8)
Test: (1000, 8)


In [8]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=5000, random_state=42))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Accuracy:", accuracy_score(y_test, y_pred_log))

Logistic Accuracy: 0.774


In [9]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:\n", classification_report(y_test, y_pred_log))

Confusion Matrix:
 [[774   3]
 [223   0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      1.00      0.87       777
           1       0.00      0.00      0.00       223

    accuracy                           0.77      1000
   macro avg       0.39      0.50      0.44      1000
weighted avg       0.60      0.77      0.68      1000



In [10]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.758


In [11]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Confusion Matrix:
 [[747  30]
 [212  11]]

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.96      0.86       777
           1       0.27      0.05      0.08       223

    accuracy                           0.76      1000
   macro avg       0.52      0.51      0.47      1000
weighted avg       0.67      0.76      0.69      1000



In [12]:
print("Logistic Accuracy:", accuracy_score(y_test, y_pred_log))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Logistic Accuracy: 0.774
Random Forest Accuracy: 0.758


In [13]:
cv_log = cross_val_score(log_model, X, y, cv=5)
cv_rf = cross_val_score(rf_model, X, y, cv=5)

print("Logistic CV:", cv_log)
print("Logistic Mean:", cv_log.mean())

print("Random Forest CV:", cv_rf)
print("Random Forest Mean:", cv_rf.mean())

Logistic CV: [0.772 0.776 0.777 0.773 0.776]
Logistic Mean: 0.7748000000000002
Random Forest CV: [0.764 0.759 0.744 0.737 0.762]
Random Forest Mean: 0.7532000000000001


In [14]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance)

                               Feature  Importance
2                                  LTV    0.281103
5                            AGE_YEARS    0.274687
1                           ASSET_COST    0.269248
0                    PERFORM_CNS_SCORE    0.104167
4                      NO_OF_INQUIRIES    0.031069
6        EMPLOYMENT_TYPE_Self employed    0.020628
3  DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS    0.014257
7              EMPLOYMENT_TYPE_Unknown    0.004841


In [21]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=5000, random_state=42))
])

log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nLogistic Regression Classification Report:\n")
print(classification_report(y_test, y_pred_log))

Logistic Regression Accuracy: 0.774

Logistic Regression Classification Report:

              precision    recall  f1-score   support

           0       0.78      1.00      0.87       777
           1       0.00      0.00      0.00       223

    accuracy                           0.77      1000
   macro avg       0.39      0.50      0.44      1000
weighted avg       0.60      0.77      0.68      1000



In [23]:
rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRandom Forest Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.758

Random Forest Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.96      0.86       777
           1       0.27      0.05      0.08       223

    accuracy                           0.76      1000
   macro avg       0.52      0.51      0.47      1000
weighted avg       0.67      0.76      0.69      1000



In [24]:
log_acc = accuracy_score(y_test, y_pred_log)
rf_acc = accuracy_score(y_test, y_pred_rf)

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [log_acc, rf_acc]
})

print(comparison_df)

                 Model  Accuracy
0  Logistic Regression     0.774
1        Random Forest     0.758
